In [1]:
import pandas as pd

In [2]:
train_samples = pd.read_csv("train_dataset_advanced.csv")
val_samples = pd.read_csv('val_dataset_advanced.csv')

In [3]:
train_samples.head(3)

,user_id,post_id,target,topic,text_len,word_count,text,gender,age,country,...,like_topic_covid,like_topic_entertainment,like_topic_movie,like_topic_politics,like_topic_sport,like_topic_tech,most_common_topic,topic_match,len_diff_from_avg_view,len_ratio_from_avg_view
0,200,1452,1,sport,3393,563,Middlesbrough 2-2 Charlton\n\nA late header by...,1,34,Russia,...,2.0,1.0,6.0,2.0,4.0,0.0,movie,0,1816.757764,2.151223
1,200,6919,1,movie,799,137,Methinks the best screen version of Quo Vadis?...,1,34,Russia,...,2.0,1.0,6.0,2.0,4.0,0.0,movie,1,-777.242236,0.506580
2,200,7143,1,movie,660,111,Heartland was in production about the same tim...,1,34,Russia,...,2.0,1.0,6.0,2.0,4.0,0.0,movie,1,-916.242236,0.418452


In [4]:
train_samples.shape, val_samples.shape

((6048228, 40), (3136970, 40))

In [5]:
from catboost import Pool, CatBoostClassifier

# Определим списки признаков
cat_features_names = ['gender', 'country', 'city', 'os', 'source', 'topic', 'most_common_topic']

# Все остальные числовые (кроме user_id, post_id, target)
feature_names = [col for col in train_samples.columns if col not in ['user_id', 'post_id', 'target', 'text']]

# Создаём Pool
train_pool = Pool(
    data=train_samples[feature_names],
    label=train_samples['target'],
    cat_features=cat_features_names,
)

val_pool = Pool(
    data=val_samples[feature_names],
    label=val_samples['target'],
    cat_features=cat_features_names,
)

print("Готово к обучению CatBoost!")

Готово к обучению CatBoost!


In [6]:
model = CatBoostClassifier(
    iterations=500,
    early_stopping_rounds=200,
    eval_metric="AUC",
    max_depth=None,
    learning_rate=0.1,
)

In [7]:
model.fit(train_pool, eval_set=val_pool, verbose=25)

0:	test: 0.5721789	best: 0.5721789 (0)	total: 1.19s	remaining: 9m 53s
25:	test: 0.6101201	best: 0.6104208 (17)	total: 21.7s	remaining: 6m 36s
50:	test: 0.6225797	best: 0.6225797 (50)	total: 47.3s	remaining: 6m 56s
75:	test: 0.6241906	best: 0.6245418 (69)	total: 1m 13s	remaining: 6m 47s
100:	test: 0.6242466	best: 0.6245418 (69)	total: 1m 37s	remaining: 6m 25s
125:	test: 0.6233107	best: 0.6245418 (69)	total: 2m 2s	remaining: 6m 3s
150:	test: 0.6234039	best: 0.6245418 (69)	total: 2m 27s	remaining: 5m 41s
175:	test: 0.6228131	best: 0.6245418 (69)	total: 2m 52s	remaining: 5m 18s
200:	test: 0.6225214	best: 0.6245418 (69)	total: 3m 16s	remaining: 4m 52s
225:	test: 0.6215210	best: 0.6245418 (69)	total: 3m 42s	remaining: 4m 29s
250:	test: 0.6206681	best: 0.6245418 (69)	total: 4m 8s	remaining: 4m 6s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.6245418439
bestIteration = 69

Shrink model to first 70 iterations.


In [8]:
pd.DataFrame({
    "name": model.feature_names_,
    "fi": model.get_feature_importance(), 
}).sort_values("fi", ascending=False)

,name,fi
6,city,21.576667
7,exp_group,19.407060
0,topic,14.162754
4,age,13.958539
15,unique_likes,7.064214
14,total_likes,4.746534
2,word_count,3.836420
11,unique_views,2.869816
3,gender,2.820522
32,most_common_topic,2.594798


In [11]:
import optuna
from catboost import CatBoostClassifier, Pool
import numpy as np

def objective(trial, train_pool, val_pool):
    # Задаём пространство поиска
    params = {
        'iterations': 2000,
        'early_stopping_rounds': 200,
        'eval_metric': 'AUC',
        'loss_function': trial.suggest_categorical("loss_function", ["Logloss", "CrossEntropy"]),
        'task_type': 'CPU',
        'thread_count': -1,
        'random_seed': 42,
        'verbose': False,
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.5),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_uniform('bagging_temperature', 0, 1),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_uniform('random_strength', 0, 2),
        'one_hot_max_size':  trial.suggest_int('one_hot_max_size', 2, 10),
    }
    
    # Обучаем модель
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, verbose_eval=False)
    
    # Извлекаем лучшее значение метрики на валидации
    best_auc = model.best_score_['validation']['AUC']
    
    return best_auc

In [12]:
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

# Создаём study с направлением максимизации AUC
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10)
)

objective_func = lambda trial: objective(trial, train_pool, val_pool)

# Оптимизация (например, 30 попыток)
study.optimize(objective_func, n_trials=30, show_progress_bar=True)

[I 2026-03-03 16:44:32,673] A new study created in memory with name: no-name-d6924ecd-693f-4d6a-ba86-81426ba24e6d
  0%|          | 0/30 [00:00<?, ?it/s]/var/folders/9q/zwzshfys2s32y6mnrh7crj1m0000gp/T/ipykernel_3888/385456407.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.5),
/var/folders/9q/zwzshfys2s32y6mnrh7crj1m0000gp/T/ipykernel_3888/385456407.py:18: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1, 10),
/var/folders/9q/zwzshfys2s32y6mnrh7crj1m0000gp/T/ipykernel_3888/385456407.py:19: FutureWarning: suggest_uniform has been deprecated in v3.

[I 2026-03-03 16:48:38,382] Trial 0 finished with value: 0.6256281908658784 and parameters: {'loss_function': 'CrossEntropy', 'learning_rate': 0.09454306819536165, 'depth': 8, 'l2_leaf_reg': 1.4322493718230251, 'bagging_temperature': 0.15599452033620265, 'border_count': 45, 'random_strength': 1.7323522915498704, 'one_hot_max_size': 7}. Best is trial 0 with value: 0.6256281908658784.


Best trial: 0. Best value: 0.625628:   7%|▋         | 2/30 [08:22<1:57:35, 251.97s/it]

[I 2026-03-03 16:52:54,731] Trial 1 finished with value: 0.6246765460607083 and parameters: {'loss_function': 'Logloss', 'learning_rate': 0.4147225000481636, 'depth': 9, 'l2_leaf_reg': 1.6305687346221471, 'bagging_temperature': 0.18182496720710062, 'border_count': 73, 'random_strength': 0.6084844859190754, 'one_hot_max_size': 6}. Best is trial 0 with value: 0.6256281908658784.


Best trial: 0. Best value: 0.625628:  10%|█         | 3/30 [16:30<2:41:56, 359.85s/it]

...

[I 2026-03-03 23:25:15,470] Trial 27 finished with value: 0.6264290935596806 and parameters: {'loss_function': 'CrossEntropy', 'learning_rate': 0.18189424440159324, 'depth': 9, 'l2_leaf_reg': 6.780723052916662, 'bagging_temperature': 0.7794727485249376, 'border_count': 60, 'random_strength': 0.8391187701157254, 'one_hot_max_size': 9}. Best is trial 24 with value: 0.6269020923088026.


Best trial: 24. Best value: 0.626902:  97%|█████████▋| 29/30 [6:45:46<06:05, 365.84s/it]

[I 2026-03-03 23:30:19,553] Trial 28 finished with value: 0.6256188028250649 and parameters: {'loss_function': 'CrossEntropy', 'learning_rate': 0.16695749465927126, 'depth': 10, 'l2_leaf_reg': 2.9799889765878853, 'bagging_temperature': 0.9032047917147669, 'border_count': 44, 'random_strength': 0.40853533406730624, 'one_hot_max_size': 9}. Best is trial 24 with value: 0.6269020923088026.


Best trial: 24. Best value: 0.626902: 100%|██████████| 30/30 [6:49:41<00:00, 819.38s/it]

[I 2026-03-03 23:34:14,006] Trial 29 finished with value: 0.6229540186852001 and parameters: {'loss_function': 'CrossEntropy', 'learning_rate': 0.4251974417106103, 'depth': 8, 'l2_leaf_reg': 3.888306735313911, 'bagging_temperature': 0.5999136507716738, 'border_count': 32, 'random_strength': 0.5881301456190623, 'one_hot_max_size': 6}. Best is trial 24 with value: 0.6269020923088026.


In [13]:
# Вывод лучших параметров
print("Best trial:")
trial = study.best_trial
print(f"AUC: {trial.value}")
print("Params:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

Best trial:
AUC: 0.6269020923088026
Params:
    loss_function: CrossEntropy
    learning_rate: 0.17491200113602548
    depth: 10
    l2_leaf_reg: 7.728745475203319
    bagging_temperature: 0.8698601875850904
    border_count: 61
    random_strength: 0.41567687837684253
    one_hot_max_size: 9


In [15]:
study.best_params

{'loss_function': 'CrossEntropy',
 'learning_rate': 0.17491200113602548,
 'depth': 10,
 'l2_leaf_reg': 7.728745475203319,
 'bagging_temperature': 0.8698601875850904,
 'border_count': 61,
 'random_strength': 0.41567687837684253,
 'one_hot_max_size': 9}

In [16]:
params = {
    'iterations': 2000,
    'early_stopping_rounds': 200,
    'eval_metric': 'AUC',
    'loss_function': study.best_params["loss_function"],
    'task_type': 'CPU',
    'thread_count': -1,
    'random_seed': 42,
    'verbose': False,
    'learning_rate': study.best_params["learning_rate"],
    'depth': study.best_params["depth"],
    'l2_leaf_reg': study.best_params["l2_leaf_reg"],
    'bagging_temperature': study.best_params["bagging_temperature"],
    'border_count': study.best_params["border_count"],
    'random_strength': study.best_params["random_strength"],
    'one_hot_max_size':  study.best_params["one_hot_max_size"],
}

# Обучаем модель
model = CatBoostClassifier(**params)
model.fit(train_pool, eval_set=val_pool, verbose=50)

0:	test: 0.5816443	best: 0.5816443 (0)	total: 1.31s	remaining: 43m 38s
50:	test: 0.6253167	best: 0.6269021 (34)	total: 1m 2s	remaining: 39m 56s
100:	test: 0.6168027	best: 0.6269021 (34)	total: 2m 9s	remaining: 40m 33s
150:	test: 0.6130649	best: 0.6269021 (34)	total: 3m 18s	remaining: 40m 27s
200:	test: 0.6107269	best: 0.6269021 (34)	total: 4m 25s	remaining: 39m 33s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.6269020923
bestIteration = 34

Shrink model to first 35 iterations.


In [17]:
model.save_model("model.cbm")